In [ ]:
# Clone the repository
!git clone https://github.com/huggingface/lerobot.git

# Move into the folder
%cd lerobot

# Install LeRobot with SmolVLA dependencies
!pip install -e ".[smolvla]"

# Install server tools AND nest-asyncio for Colab compatibility
!pip install fastapi uvicorn pyngrok pydantic nest-asyncio

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Authenticate your ngrok tunnel (Requires adding 'ngrok_auth_token' to Colab Secrets)
ngrok.set_auth_token(userdata.get('ngrok_auth_token'))

# Open a tunnel on port 8000
public_url = ngrok.connect(8000).public_url
print(f"--> SUCCESS! NGROK PUBLIC URL: {public_url}")

In [ ]:
import sys
import os
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import torch
import base64
import numpy as np
from io import BytesIO
from PIL import Image

# 1. Apply the async patch so Uvicorn can run inside Colab
nest_asyncio.apply()

# 2. Hardcode the path so Python finds the cloned lerobot library
sys.path.insert(0, "/content/lerobot/src")
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

app = FastAPI()

# 3. Target the T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Server] Loading SmolVLA on {device}...")

# 4. Load the policy in Half-Precision (bfloat16) to save VRAM and speed up inference
policy = SmolVLAPolicy.from_pretrained(
    "lerobot/smolvla_base",
    torch_dtype=torch.float32
)

policy.config.num_steps = 30
policy.use_cache = False

policy.to(device)
policy.eval()

class InferenceRequest(BaseModel):
    image1: str
    image3: str
    instruction: str

@app.post("/predict")
def predict(request: InferenceRequest):
    try:
        img1_data = base64.b64decode(request.image1)
        image1 = Image.open(BytesIO(img1_data)).convert("RGB")

        img3_data = base64.b64decode(request.image3)
        image3 = Image.open(BytesIO(img3_data)).convert("RGB")

        # Convert to Tensor [3, H, W]
        img1_tensor = torch.from_numpy(np.array(image1)).permute(2, 0, 1).float() / 255.0
        img3_tensor = torch.from_numpy(np.array(image3)).permute(2, 0, 1).float() / 255.0

        # Handle Tokenization
        tokenized_text = policy.model.vlm_with_expert.processor.tokenizer(
            request.instruction,
            return_tensors="pt",
            padding=True
        )

        # Build the observation batch
        observation = {
            "observation.images.camera1": img1_tensor.unsqueeze(0).to(device),
            "observation.images.camera3": img3_tensor.unsqueeze(0).to(device),
            "observation.state": torch.zeros((1, 6)).to(device),
            "observation.language.tokens": tokenized_text.input_ids.to(device),
            "observation.language.attention_mask": tokenized_text.attention_mask.to(device).bool()
        }

        # Inference
        with torch.no_grad():
            action_chunk = policy.select_action(observation)

        # Return first action: [dx, dy, dz, rx, ry, rz, gripper]
        # next_action = action_chunk[0].cpu().numpy().tolist()
        # return {"status": "success", "action": next_action}

        # Return all actions
        actions = action_chunk.cpu().numpy().tolist()
        return {"status": "success", "actions": actions}

    except Exception as e:
        import traceback
        print(f" -> [Server Side Error] {str(e)}")
        traceback.print_exc()
        return {"status": "error", "message": str(e)}

# 5. Start the server in a separate background thread
import threading
import asyncio

def run_server():
    # Create a brand new event loop specifically for this thread
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()

print("Starting FastAPI server in the background...")
threading.Thread(target=run_server, daemon=True).start()
print("Server is live 🎉 You can now send requests from Webots.")